# Path-Consistent Energy Model (PCEM)

This notebook trains and evaluates a physics-anchored endpoint scorer:

$$
\log s(e \mid \mathrm{traj}) = \log p_{\mathrm{physics}}(e \mid \mathrm{traj}) + \lambda \, \Delta_{\mathrm{nn}}(e, \mathrm{traj}).
$$

The physics term is the exact synthetic magnetic likelihood from `classic.ipynb`. The neural correction starts at zero, so the model begins near the classic baseline and can only improve it.

Evaluation uses the same finite endpoint grid as the classic estimator:

$$
I(e; \mathrm{traj}) = H(e) - H(e \mid \mathrm{traj}).
$$

## 0. Kaggle: clone repository

Run this cell **only on Kaggle** (Settings → Accelerator: **GPU**, Internet: **On**).

Repository: [romanbranovets/inverse_positioning](https://github.com/romanbranovets/inverse_positioning)

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/romanbranovets/inverse_positioning.git"
REPO_DIR = Path("/kaggle/working/inverse_positioning")

if Path("/kaggle/input").exists():
    if not (REPO_DIR / "notebooks" / "shared").exists():
        if REPO_DIR.exists():
            subprocess.run(["rm", "-rf", str(REPO_DIR)], check=True)
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            check=True,
        )
    os.chdir(REPO_DIR / "notebooks")
    print(f"Kaggle mode: repo at {REPO_DIR}")
else:
    print("Local mode: skip clone (not on Kaggle)")

## 1. Setup

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import clear_output, display
from torch.nn import functional as F
from tqdm.auto import tqdm

NOTEBOOK_ROOT = Path.cwd()
SEARCH_ROOTS = [
    NOTEBOOK_ROOT,
    NOTEBOOK_ROOT / "notebooks",
    Path("/kaggle/working/inverse_positioning/notebooks"),
]
for root in SEARCH_ROOTS:
    if (root / "shared").exists():
        sys.path.insert(0, str(root))
        print(f"Using shared from: {root / 'shared'}")
        break
else:
    raise FileNotFoundError(
        "Could not find notebooks/shared. On Kaggle, run the git clone cell first."
    )

from shared.pcem import *
from shared.sphere_utils import fibonacci_sphere
from shared.trajectory_sampler import estimate_feature_normalization, make_trajectories

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

FEATURE_MEAN, FEATURE_STD = estimate_feature_normalization(device=DEVICE)

ENDPOINT_H_BITS = 14
ENDPOINT_COUNT = 2**ENDPOINT_H_BITS
EVAL_STEP_COUNTS = [1, 2, 4, 8, 12, 16]
FOCUS_STEP_COUNT = 16

D_MODEL = 256
N_HEADS = 4
N_LAYERS = 3
DROPOUT = 0.05
NEURAL_WEIGHT = 1.0
NEURAL_REG_WEIGHT = 0.01

TRAIN_EPOCHS = 120
TRAIN_BATCHES_PER_EPOCH = 6
TRAIN_BATCH_SIZE = 128
TRAIN_NEGATIVES = 31
LEARNING_RATE = 2e-4
PLOT_EVERY = 5

EVAL_TRAJECTORIES = 64
EVAL_GRID_CHUNK = 512
NWJ_NEGATIVES = 32
NWJ_BATCHES = 16

print(
    f"device={DEVICE}, cuda={torch.cuda.is_available()}, "
    f"endpoint grid={ENDPOINT_COUNT} points, H(e)={ENDPOINT_H_BITS:.1f} bits"
)

## 2. Endpoint Grid And Model

In [ ]:
endpoint_grid = fibonacci_sphere(ENDPOINT_COUNT, device=DEVICE)


def create_pcem_bundle():
    model = PathConsistentEnergyModel(
        feature_dim=FEATURE_DIM,
        feature_mean=FEATURE_MEAN,
        feature_std=FEATURE_STD,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        max_tokens=17,
        dropout=DROPOUT,
        neural_weight=NEURAL_WEIGHT,
    ).to(device=DEVICE, dtype=DTYPE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=TRAIN_EPOCHS * TRAIN_BATCHES_PER_EPOCH,
        eta_min=0.1 * LEARNING_RATE,
    )
    return model, optimizer, scheduler


_parameter_count = PathConsistentEnergyModel(
    feature_dim=FEATURE_DIM,
    feature_mean=FEATURE_MEAN,
    feature_std=FEATURE_STD,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
).to(device=DEVICE, dtype=DTYPE)
sum(p.numel() for p in _parameter_count.parameters())

## 3. Training

Each batch uses the true grid endpoint plus uniform-sphere negatives. The loss is cross-entropy over candidate scores plus a small penalty that keeps the neural correction near zero early in training.

In [ ]:
def train_pcem_for_steps(step_count):
    model, optimizer, scheduler = create_pcem_bundle()
    history = []
    model.train()

    for epoch in range(1, TRAIN_EPOCHS + 1):
        epoch_losses = []
        epoch_accuracies = []
        epoch_neural = []
        for _ in range(TRAIN_BATCHES_PER_EPOCH):
            x, pad_mask, endpoint, _ = make_trajectories(
                TRAIN_BATCH_SIZE, step_counts=step_count
            )
            candidates, labels = make_candidate_batch(
                endpoint, TRAIN_NEGATIVES, device=DEVICE
            )
            scores = model(candidates, x, pad_mask)
            ce_loss = F.cross_entropy(scores, labels)
            neural = model.neural_correction(
                model.encode_trajectory(x, pad_mask), candidates, x, pad_mask
            )
            reg_loss = NEURAL_REG_WEIGHT * neural.square().mean()
            loss = ce_loss + reg_loss

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            epoch_losses.append(float(loss.detach().cpu()))
            epoch_accuracies.append(
                float((scores.argmax(dim=1) == labels).float().mean().cpu())
            )
            epoch_neural.append(float(neural.abs().mean().cpu()))

        row = dict(
            epoch=epoch,
            loss=float(np.mean(epoch_losses)),
            accuracy=float(np.mean(epoch_accuracies)),
            neural_abs=float(np.mean(epoch_neural)),
        )
        history.append(row)
        if epoch == 1 or epoch % PLOT_EVERY == 0 or epoch == TRAIN_EPOCHS:
            clear_output(wait=True)
            fig = go.Figure()
            fig.add_trace(
                go.Scatter(
                    x=[item["epoch"] for item in history],
                    y=[item["loss"] for item in history],
                    mode="lines",
                    name="train loss",
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=[item["epoch"] for item in history],
                    y=[item["accuracy"] for item in history],
                    mode="lines",
                    name="candidate accuracy",
                    yaxis="y2",
                )
            )
            fig.update_layout(
                title=f"Training PCEM, steps={step_count}, epoch {epoch}/{TRAIN_EPOCHS}",
                xaxis_title="epoch",
                yaxis_title="loss",
                yaxis2=dict(title="accuracy", overlaying="y", side="right"),
                height=360,
                margin=dict(l=40, r=20, t=50, b=40),
            )
            display(fig)

    return {"model": model, "history": history, "step_count": step_count}


trained_models = {}
for steps in EVAL_STEP_COUNTS:
    trained_models[int(steps)] = train_pcem_for_steps(int(steps))

for steps, result in trained_models.items():
    last = result["history"][-1]
    print(
        f"steps={steps:2d}: loss={last['loss']:.4f}, "
        f"acc={last['accuracy']:.4f}, neural_abs={last['neural_abs']:.4f}"
    )

## 4. Grid Posterior Evaluation

For each sampled grid trajectory we score every endpoint candidate, build the posterior table, and estimate

$$
I(e; \mathrm{traj}) = H(e) - H(e \mid \mathrm{traj}).
$$

We report both the physics-only scorer and the full PCEM scorer.

In [ ]:
@torch.no_grad()
def estimate_grid_mi(model, step_count, trajectory_count, physics_only=False):
    model.eval()
    mi_samples = []
    h_cond_samples = []
    iterator = tqdm(
        range(trajectory_count),
        desc=(
            f"grid MI, steps={step_count}, "
            f"{'physics' if physics_only else 'pcem'}"
        ),
    )
    for _ in iterator:
        x, pad_mask, _, _, _ = sample_grid_trajectory(
            endpoint_grid, step_count, batch_size=1
        )
        log_scores = score_endpoint_grid(
            model,
            endpoint_grid,
            x,
            pad_mask,
            chunk_size=EVAL_GRID_CHUNK,
            physics_only=physics_only,
        )
        _, h_cond_bits = posterior_from_grid_scores(log_scores, ENDPOINT_COUNT)
        h_cond_samples.append(h_cond_bits)
        mi_samples.append(ENDPOINT_H_BITS - h_cond_bits)
        iterator.set_postfix(mi_bits=f"{mi_samples[-1]:.2f}")
    mi = np.asarray(mi_samples, dtype=np.float64)
    h_cond = np.asarray(h_cond_samples, dtype=np.float64)
    return {
        "steps": int(step_count),
        "distance_m": float(step_count * STEP_METERS),
        "mi_bits_mean": float(mi.mean()),
        "mi_bits_std": float(mi.std(ddof=1)) if mi.size > 1 else 0.0,
        "h_cond_bits_mean": float(h_cond.mean()),
        "samples": int(mi.size),
        "physics_only": bool(physics_only),
    }


grid_rows = []
for steps in EVAL_STEP_COUNTS:
    grid_rows.append(
        estimate_grid_mi(
            trained_models[steps]["model"],
            steps,
            EVAL_TRAJECTORIES,
            physics_only=True,
        )
    )
    grid_rows.append(
        estimate_grid_mi(
            trained_models[steps]["model"],
            steps,
            EVAL_TRAJECTORIES,
            physics_only=False,
        )
    )

for row in grid_rows:
    label = "physics" if row["physics_only"] else "pcem   "
    print(
        f"{label}, steps={row['steps']:2d}, distance={row['distance_m']:6.0f} m, "
        f"MI={row['mi_bits_mean']:.4f} +/- {row['mi_bits_std']:.4f} bits, "
        f"h_cond={row['h_cond_bits_mean']:.4f} bits"
    )

## 5. NWJ Audit On The Focus Model

The same score function can be used as an NWJ critic. Because the physics term dominates, this audit is usually much more stable than a pure neural discriminator at long trajectories.

In [ ]:
@torch.no_grad()
def estimate_pcem_nwj(
    model, step_count, batches, batch_size, negative_count, physics_only=False
):
    model.eval()
    batch_values = []
    for _ in range(batches):
        x, pad_mask, endpoint, _ = make_trajectories(
            batch_size, step_counts=step_count
        )
        positive, negative = contrastive_nwj_scores(
            model,
            x,
            pad_mask,
            endpoint,
            negative_count,
            negative_chunk_size=256,
            physics_only=physics_only,
        )
        batch_values.append(
            nwj_bits_from_scores(positive.reshape(-1), negative.reshape(-1))
        )
    return float(np.mean(batch_values))


focus_model = trained_models[FOCUS_STEP_COUNT]["model"]
for eval_steps in EVAL_STEP_COUNTS:
    physics_nwj = estimate_pcem_nwj(
        focus_model,
        eval_steps,
        NWJ_BATCHES,
        TRAIN_BATCH_SIZE,
        NWJ_NEGATIVES,
        physics_only=True,
    )
    full_nwj = estimate_pcem_nwj(
        focus_model,
        eval_steps,
        NWJ_BATCHES,
        TRAIN_BATCH_SIZE,
        NWJ_NEGATIVES,
        physics_only=False,
    )
    print(
        f"trained={FOCUS_STEP_COUNT:2d}, eval={eval_steps:2d}, "
        f"distance={eval_steps * STEP_METERS:6.0f} m, "
        f"NWJ_physics={physics_nwj:.4f} bits, NWJ_full={full_nwj:.4f} bits"
    )

## 6. Example Posterior On One Trajectory

In [ ]:
@torch.no_grad()
def example_posterior(model, step_count):
    model.eval()
    x, pad_mask, endpoint, endpoint_index, path = sample_grid_trajectory(
        endpoint_grid, step_count, batch_size=1
    )
    physics_scores = score_endpoint_grid(
        model, endpoint_grid, x, pad_mask, physics_only=True
    )
    pcem_scores = score_endpoint_grid(
        model, endpoint_grid, x, pad_mask, physics_only=False
    )
    physics_posterior, physics_h = posterior_from_grid_scores(
        physics_scores, ENDPOINT_COUNT
    )
    pcem_posterior, pcem_h = posterior_from_grid_scores(
        pcem_scores, ENDPOINT_COUNT
    )
    true_idx = int(endpoint_index[0].cpu())
    return dict(
        steps=step_count,
        endpoint=endpoint[0].detach().cpu(),
        path=path[0].detach().cpu(),
        true_index=true_idx,
        physics_mi=ENDPOINT_H_BITS - physics_h,
        pcem_mi=ENDPOINT_H_BITS - pcem_h,
        physics_true_prob=float(physics_posterior[true_idx].cpu()),
        pcem_true_prob=float(pcem_posterior[true_idx].cpu()),
    )


example = example_posterior(trained_models[FOCUS_STEP_COUNT]["model"], FOCUS_STEP_COUNT)
print(
    f"steps={example['steps']}, physics MI={example['physics_mi']:.3f} bits, "
    f"PCEM MI={example['pcem_mi']:.3f} bits"
)
print(
    f"true endpoint prob: physics={example['physics_true_prob']:.3e}, "
    f"pcem={example['pcem_true_prob']:.3e}"
)

## 7. MI Curve Plot

In [ ]:
physics_rows = [row for row in grid_rows if row["physics_only"]]
pcem_rows = [row for row in grid_rows if not row["physics_only"]]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=[row["distance_m"] for row in physics_rows],
        y=[row["mi_bits_mean"] for row in physics_rows],
        error_y=dict(
            type="data",
            array=[row["mi_bits_std"] for row in physics_rows],
            visible=True,
        ),
        mode="lines+markers",
        name="physics only",
    )
)
fig.add_trace(
    go.Scatter(
        x=[row["distance_m"] for row in pcem_rows],
        y=[row["mi_bits_mean"] for row in pcem_rows],
        error_y=dict(
            type="data",
            array=[row["mi_bits_std"] for row in pcem_rows],
            visible=True,
        ),
        mode="lines+markers",
        name="PCEM",
    )
)
fig.update_layout(
    title="Grid MI estimate: physics baseline vs PCEM",
    xaxis_title="walk distance, meters",
    yaxis_title="I(endpoint; trajectory), bits",
    height=420,
    margin=dict(l=40, r=20, t=50, b=40),
)
fig